In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

sns.set_theme(style="whitegrid")

In [4]:

df = pd.read_excel('../data/raw/Telco_customer_churn.xlsx')
print("\n--- Data Shape ---")
display(df.shape)

print("\n--- Data Information ---")
df.info()

print("\n--- Data Sample ---")
display(df.head())


--- Data Shape ---


(7043, 33)


--- Data Information ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null   object 
 16

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9058-HRZSV,1,United States,California,Baldwin Park,91706,"34.098275, -117.967399",34.098275,-117.967399,Female,...,Month-to-month,No,Electronic check,94.40,6126.15,No,0,7,5348,NaN
2,4767-HZZHQ,1,United States,California,Redondo Beach,90277,"33.830453, -118.384565",33.830453,-118.384565,Male,...,Month-to-month,No,Bank transfer (automatic),82.05,2570.2,No,0,9,5036,NaN
3,8734-DKSTZ,1,United States,California,Winterhaven,92283,"32.852947, -114.850784",32.852947,-114.850784,Female,...,Month-to-month,No,Electronic check,85.95,858.6,No,0,20,4922,NaN
4,0913-XWSCN,1,United States,California,Sun City,92585,"33.739412, -117.173334",33.739412,-117.173334,Male,...,Month-to-month,No,Bank transfer (automatic),85.50,4713.4,No,0,20,6411,NaN


In [ ]:
# Null Investigation
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
null_summary = pd.DataFrame({
    "null_count": null_counts,
    "null_pct": null_pct,
}).sort_values("null_count", ascending=False)

print(f"Total columns: {df.shape[1]} | Columns with nulls: {(null_counts > 0).sum()}")
display(null_summary[null_summary["null_count"] > 0])

# Bar chart of columns with missing values
cols_with_nulls = null_summary[null_summary["null_count"] > 0]
if not cols_with_nulls.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(cols_with_nulls.index, cols_with_nulls["null_count"],
                  color=sns.color_palette("Set2", len(cols_with_nulls)))
    ax.set_title("Missing Values per Column")
    ax.set_ylabel("Null Count")
    ax.set_ylim(0, cols_with_nulls["null_count"].max() * 1.18)
    ax.tick_params(axis="x", labelrotation=30)
    for bar in bars:
        h = bar.get_height()
        ax.annotate(
            f"{h:,} ({h / len(df):.1%})",
            xy=(bar.get_x() + bar.get_width() / 2, h),
            xytext=(0, 4), textcoords="offset points",
            ha="center", va="bottom", fontsize=9,
        )
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found.")

In [5]:
city_counts = df["City"].value_counts(dropna=False)
distinct_city_count = df["City"].nunique(dropna=False)

print(f"Distinct cities: {distinct_city_count:,}")
display(city_counts.rename("customer_count").to_frame())

Distinct cities: 1,129


,customer_count
City,
Los Angeles,305
San Diego,150
San Jose,112
Sacramento,108
San Francisco,104
...,...
Douglas City,4
Canyon Dam,4
Ludlow,4


In [ ]:
#Chunk number 4
cat_cols = [
    "Gender", "Senior Citizen", "Partner", "Dependents",
    "Phone Service", "Multiple Lines", "Internet Service",
    "Online Security", "Online Backup", "Device Protection",
    "Tech Support", "Streaming TV", "Streaming Movies",
    "Contract", "Paperless Billing", "Payment Method",
]

n_cols = 4
n_rows = -(-len(cat_cols) // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    counts = df[col].value_counts()
    total = counts.sum()
    bars = axes[i].bar(counts.index.astype(str), counts.values, color=sns.color_palette("Set2", len(counts)))
    axes[i].set_title(col)
    axes[i].set_ylabel("Count")
    axes[i].set_ylim(0, counts.max() * 1.18)
    axes[i].tick_params(axis="x", labelrotation=30)
    axes[i].set_xticklabels(counts.index.astype(str), ha="right")
    for bar in bars:
        height = bar.get_height()
        axes[i].annotate(
            f"{height:,}\n({height / total:.1%})",
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center", va="bottom",
            fontsize=8,
        )

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
#Chunk number 5
num_cols = ["Tenure Months", "Monthly Charges", "Total Charges", "CLTV"]

display(df[num_cols].apply(pd.to_numeric, errors="coerce").describe())

# Boxplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    data = pd.to_numeric(df[col], errors="coerce").dropna()
    bp = axes[i].boxplot(
        data,
        patch_artist=True,
        showfliers=True,
        boxprops=dict(facecolor=sns.color_palette("Set2")[i], alpha=0.8),
        medianprops=dict(color="black", linewidth=2),
        flierprops=dict(marker="o", markerfacecolor="red", markersize=3, alpha=0.4, linestyle="none"),
    )
    axes[i].set_title(col)
    axes[i].set_ylabel("Value")
    axes[i].set_xticks([])

    x_right = bp["boxes"][0].get_path().vertices[:, 0].max()
    q1, median, q3 = data.quantile([0.25, 0.50, 0.75])

    for whisker in bp["whiskers"]:
        yval = whisker.get_ydata()[1]
        axes[i].annotate(
            f"{yval:,.1f}",
            xy=(x_right, yval), xytext=(6, 0), textcoords="offset points",
            ha="left", va="center", fontsize=9,
        )
    for yval in (q1, q3):
        axes[i].annotate(
            f"{yval:,.1f}",
            xy=(x_right, yval), xytext=(6, 0), textcoords="offset points",
            ha="left", va="center", fontsize=9,
        )
    axes[i].annotate(
        f"{median:,.1f}",
        xy=(x_right, median), xytext=(6, 0), textcoords="offset points",
        ha="left", va="center", fontsize=9, fontweight="bold",
    )

plt.tight_layout()
plt.show()

# Histograms
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    data = pd.to_numeric(df[col], errors="coerce").dropna()
    axes[i].hist(data, bins=30, color=sns.color_palette("Set2")[i], edgecolor="white", alpha=0.85)
    axes[i].set_title(f"{col} — Distribution")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# Z-Score Outlier Detection (|Z| > 3)
num_cols = ["Tenure Months", "Monthly Charges", "Total Charges", "CLTV"]

num_df = df[num_cols].apply(pd.to_numeric, errors="coerce")

z_scores = (num_df - num_df.mean()) / num_df.std()
outlier_mask = z_scores.abs() > 3

summary = pd.DataFrame({
    "outlier_count": outlier_mask.sum(),
    "outlier_pct": (outlier_mask.sum() / len(num_df) * 100).round(2),
    "mean": num_df.mean().round(2),
    "std": num_df.std().round(2),
    "z_threshold_lower": (num_df.mean() - 3 * num_df.std()).round(2),
    "z_threshold_upper": (num_df.mean() + 3 * num_df.std()).round(2),
})
print("Outlier summary (|Z| > 3):")
display(summary)

# Flag column on the full dataframe
outlier_flags = outlier_mask.any(axis=1)
print(f"\nRows flagged as outlier in at least one column: {outlier_flags.sum()} ({outlier_flags.mean():.1%})")
display(num_df[outlier_flags].describe())

# Visualise Z-scores per column
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    z = z_scores[col].dropna()
    color = sns.color_palette("Set2")[i]
    axes[i].scatter(range(len(z)), z, s=5, alpha=0.4, color=color, label="Normal")
    outliers = z[z.abs() > 3]
    axes[i].scatter(outliers.index, outliers, s=20, color="red", alpha=0.7, label=f"Outliers ({len(outliers)})")
    axes[i].axhline(3,  color="red", linestyle="--", linewidth=1)
    axes[i].axhline(-3, color="red", linestyle="--", linewidth=1)
    axes[i].set_title(f"{col} — Z-Score")
    axes[i].set_ylabel("Z-Score")
    axes[i].set_xlabel("Row index")
    axes[i].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
#Chunk number 6
display(df[["Churn Label", "Churn Score"]].describe(include="all"))

# Bar chart + Churn Score boxplot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

counts = df["Churn Label"].value_counts()
total = counts.sum()
bars = ax1.bar(counts.index.astype(str), counts.values, color=sns.color_palette("Set2", len(counts)))
ax1.set_title("Churn Label")
ax1.set_ylabel("Count")
ax1.set_ylim(0, counts.max() * 1.18)
for bar in bars:
    h = bar.get_height()
    ax1.annotate(
        f"{h:,}\n({h / total:.1%})",
        xy=(bar.get_x() + bar.get_width() / 2, h),
        xytext=(0, 4), textcoords="offset points",
        ha="center", va="bottom", fontsize=9,
    )

no_scores  = pd.to_numeric(df.loc[df["Churn Label"] == "No",  "Churn Score"], errors="coerce").dropna()
yes_scores = pd.to_numeric(df.loc[df["Churn Label"] == "Yes", "Churn Score"], errors="coerce").dropna()

palette = sns.color_palette("Set2", 2)
bp = ax2.boxplot(
    [no_scores, yes_scores],
    patch_artist=True,
    showfliers=True,
    boxprops=dict(alpha=0.8),
    medianprops=dict(color="black", linewidth=2),
    flierprops=dict(marker="o", markerfacecolor="red", markersize=3, alpha=0.4, linestyle="none"),
)
for patch, color in zip(bp["boxes"], palette):
    patch.set_facecolor(color)

ax2.set_title("Churn Score by Churn Label")
ax2.set_ylabel("Churn Score")
ax2.set_xticks([1, 2])
ax2.set_xticklabels(["Churn = No", "Churn = Yes"])

for j, data in enumerate([no_scores, yes_scores], start=1):
    x_right = bp["boxes"][j - 1].get_path().vertices[:, 0].max()
    q1, median, q3 = data.quantile([0.25, 0.50, 0.75])
    for whisker in [bp["whiskers"][(j - 1) * 2], bp["whiskers"][(j - 1) * 2 + 1]]:
        yval = whisker.get_ydata()[1]
        ax2.annotate(f"{yval:,.1f}", xy=(x_right, yval), xytext=(6, 0),
                     textcoords="offset points", ha="left", va="center", fontsize=9)
    for yval in (q1, q3):
        ax2.annotate(f"{yval:,.1f}", xy=(x_right, yval), xytext=(6, 0),
                     textcoords="offset points", ha="left", va="center", fontsize=9)
    ax2.annotate(f"{median:,.1f}", xy=(x_right, median), xytext=(6, 0),
                 textcoords="offset points", ha="left", va="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()

# Histograms — Churn Score split by Churn Label
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, (label, data), color in zip(axes, [("No", no_scores), ("Yes", yes_scores)], palette):
    ax.hist(data, bins=30, color=color, edgecolor="white", alpha=0.85)
    ax.set_title(f"Churn Score Distribution — Churn Label = {label}")
    ax.set_xlabel("Churn Score")
    ax.set_ylabel("Count")

plt.tight_layout()
plt.show()